<a href="https://colab.research.google.com/github/Hasan-ari/chemical_thesis_repo/blob/main/conditional_model_v3_8_rocks_wd/colab_eight_rocks_wd_loro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sekiz Kayaç v3: Weight Decay ile LORO

**Soru:** ağırlıkların büyümesini engellemek (weight decay) modelin **görmediği kayaca** genellemesini iyileştiriyor mu?

**Yöntem:** v2'deki LORO deneyi (8 kat, 8 model) iki yeni "yay sertliği" ile aynen tekrarlanır:

| Nokta | Config | weight_decay | Durum |
|---|---|---|---|
| 0 | `loro_colab_eight_rocks.yaml` (v2) | 0.0 | Drive'da hazır, yeniden koşulmaz |
| 1 | `loro_colab_eight_rocks_wd_1e-4.yaml` | 0.0001 | bu defter |
| 2 | `loro_colab_eight_rocks_wd_1e-3.yaml` | 0.001 | bu defter |

Başka hiçbir şey değişmez: aynı önbellek (`eight_rocks_v1`, 55.122 koşu), aynı model (hidden 128, 2 katman, dropout 0), aynı eğitim (100 epoch, batch 64, lr 1e-3, seed 42), aynı bölme oranları. Kayaç adı modele girmez.

**Bu defterde zip açma yok.** Önbellek v2 defterinde kurulup Drive'a yazıldı; buradan sadece kopyalanıp doğrulanır. Ham TXT'ler gerekmez.

**Sonuç okuma:** her kayaç için `unseen` normalize RMSE üç sertlikte yan yana. 1.0 = "ortalamayı söyleyen model" seviyesi. Başarı ölçütleri deney öncesinde yazıldı (`daily-sessions/2026-09-17_0112.md`):
- B açığı kapandı: unseen normalize RMSE 8 kayacın en az 5'inde v2'ye göre düştü **ve** en az 4'ünde 1.0'ın altına indi **ve** seen normalize RMSE 0.10'u geçmedi.
- Kapanmadı: unseen çoğunlukla ≥ 1.0 kaldı → sorun ezber değil, kapsam.


## 0. Repoyu klonla

Her oturumda taze klon. Kod GitHub'da, veri Drive'da.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# Important: leave the repo folder before deleting it.
os.chdir("/content")

scheme = "https://"
host = "github" + ".com"
owner = "Hasan-ari"
repo = "chemical_thesis_repo"

REPO_URL = f"{scheme}{host}/{owner}/{repo}.git"
BRANCH = "main"
REPO_DIR = Path("/content/chemical_thesis_repo")

print("cwd before clone:", os.getcwd())
print("REPO_URL repr:", repr(REPO_URL))

shutil.rmtree(REPO_DIR, ignore_errors=True)

result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    cwd="/content",
    capture_output=True,
    text=True,
)

print("returncode:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("git clone failed")

os.chdir(REPO_DIR)

print("cwd after clone:", os.getcwd())

print("HEAD:")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

print("Check data.py:")
subprocess.run(["ls", "-l", "conditional_model_v1/data.py"], check=True)

## 1. Ayarlar

Sadece burayı düzenle. Zip yok; önbellek Drive'daki `processed/eight_rocks_v1`.

In [ ]:
from __future__ import annotations

from pathlib import Path

REPO_DIR = Path("/content/chemical_thesis_repo")

# ---- Drive ----
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/chemical_thesis_data")
DRIVE_PROCESSED_ROOT = DRIVE_DATA_DIR / "processed"
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/chemical_thesis_repo/runs")

# ---- Colab yerel klasörler (hızlı disk) ----
DATA_ROOT = Path("/content/data")          # sadece ${DATA_ROOT} için; klasörün var olması gerekmez
PROCESSED_ROOT = Path("/content/processed")
RUN_ROOT = Path("/content/runs")

# ---- Önbellek: v2 defterinin Drive'a yazdığı eight_rocks_v1 ----
CACHE_NAME = "eight_rocks_v1"

# ---- Yay sertlikleri: etiket -> config (repo içinde; deney = config + commit) ----
WD_CONFIGS = {
    "1e-4": "configs/conditional_model_v1/loro_colab_eight_rocks_wd_1e-4.yaml",
    "1e-3": "configs/conditional_model_v1/loro_colab_eight_rocks_wd_1e-3.yaml",
}
# Bu oturumda koşulacaklar. Oturum koparsa biten sertliği listeden çıkar, kalanla devam et.
# 1e-4 17 Eylül'de bitti (aşağıda DONE_LORO_DIRS); 1e-3 o oturumda 5. katta koptu, baştan koşulur.
WD_TO_RUN = ["1e-3"]

# ---- v2 sonuçları (Drive'da bitmiş koşular) ----
# 8-kayaç referans koşu: LORO tablolarındaki `reference_*` sütunları buradan gelir.
# None bırakılırsa LORO yine koşar, sadece referans sütunları boş kalır.
REFERENCE_RUN_DIR = DRIVE_RUN_ROOT / "20260903_135204_eight_rocks_condition_lstm_v1"
# v2 LORO toplu klasörü (weight_decay 0.0): karşılaştırma tablosunun 0 noktası.
V2_LORO_DIR = DRIVE_RUN_ROOT / "20260903_141752_eight_rocks_loro_v1_loro"
# Önceki oturumlarda biten sertlikler: Drive'daki toplu klasör. Yeniden koşulmaz, sonuç tablosuna eklenir.
DONE_LORO_DIRS = {
    "1e-4": DRIVE_RUN_ROOT / "20260917_105240_eight_rocks_loro_wd_1e-4_loro",
}

# ---- Kat alt kümesi ----
# None -> config'deki held_out_rocks ("all" = 8 kat). Karar: 8 katın hepsi.
HELD_OUT_ROCKS = None

for tag, cfg in WD_CONFIGS.items():
    print(f"wd {tag}: {cfg}")
print("WD_TO_RUN:", WD_TO_RUN)
print("V2_LORO_DIR:", V2_LORO_DIR)
print("DONE_LORO_DIRS:", DONE_LORO_DIRS)


## 2. Drive'ı bağla

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Bağımlılıklar

In [ ]:
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


## 4. Ortam değişkenleri, kayıt defteri ve GPU

YAML'lar `${DATA_ROOT}` gibi değişkenler kullanır; önbellek doğrulaması ham yollara bakmadığı için `DATA_ROOT` klasörünün var olması gerekmez. Drive'daki `summary.csv` + `registry.sqlite` devam ettirilir (geçmiş silinmez).

In [ ]:
os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["PROCESSED_ROOT"] = str(PROCESSED_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)

RUN_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)

for registry_name in ("summary.csv", "registry.sqlite"):
    drive_registry = DRIVE_RUN_ROOT / registry_name
    local_registry = RUN_ROOT / registry_name
    if drive_registry.exists() and not local_registry.exists():
        shutil.copy2(drive_registry, local_registry)
        print("Drive'dan alındı:", registry_name)

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 5. Önbelleği Drive'dan al ve doğrula

TXT okuma yok. `_SUCCESS` işaretli `eight_rocks_v1` kopyalanır ve `cache validate --deep` ile her dosyanın SHA-256'sı kontrol edilir. Drive'da yoksa hücre durur.

In [ ]:
LOCAL_CACHE = PROCESSED_ROOT / CACHE_NAME
DRIVE_CACHE = DRIVE_PROCESSED_ROOT / CACHE_NAME
SUCCESS_MARKER = "_SUCCESS"
cache_command = [sys.executable, "-m", "conditional_model_v1.cli.cache"]
validate_config = WD_CONFIGS[WD_TO_RUN[0]]  # datasets + cache_name v2 ile aynı; strateji önemsiz

if not (DRIVE_CACHE / SUCCESS_MARKER).is_file():
    raise FileNotFoundError(
        f"Drive'da doğrulanmış önbellek yok: {DRIVE_CACHE}\n"
        "Bu defter önbellek kurmaz. Önce v2 defterinin 6. hücresini çalıştır "
        "(conditional_model_v2_8_rocks/colab_eight_rocks_loro.ipynb)."
    )

shutil.rmtree(LOCAL_CACHE, ignore_errors=True)
LOCAL_CACHE.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_CACHE, LOCAL_CACHE)
subprocess.run(cache_command + ["validate", "--config", validate_config, "--deep"], check=True)
print(f"Drive önbelleği kopyalandı ve doğrulandı: {DRIVE_CACHE}")


## 6. Yardımcılar ve v2 sonuçlarının kontrolü

Her kat bitince Drive'a kopyalanır; Colab kopsa bile biten katlar kaybolmaz. v2 LORO özeti (`V2_LORO_DIR`) karşılaştırma tablosunun 0 noktasıdır.

In [ ]:
def copy_run_to_drive(run_dir: Path) -> Path:
    """Koşu klasörü + summary.csv + registry.sqlite Drive'a. Var olan koşu ezilmez."""
    drive_run_dir = DRIVE_RUN_ROOT / run_dir.name
    if drive_run_dir.exists():
        raise FileExistsError(f"Drive'da aynı adlı koşu var, üzerine yazılmadı: {drive_run_dir}")
    shutil.copytree(run_dir, drive_run_dir)
    for registry_name in ("summary.csv", "registry.sqlite"):
        if (RUN_ROOT / registry_name).exists():
            shutil.copy2(RUN_ROOT / registry_name, DRIVE_RUN_ROOT / registry_name)
    print(f"Drive'a kopyalandı: {drive_run_dir}")
    return drive_run_dir


def run_loro(config_path: str) -> tuple[Path, list[Path]]:
    """cli.loro'yu çalıştır; her kat bitince Drive'a kopyala; (toplu klasör, kat klasörleri) döndür."""
    command = [sys.executable, "-m", "conditional_model_v1.cli.loro", "--config", config_path]
    if REFERENCE_RUN_DIR is not None:
        command += ["--reference-run-dir", str(REFERENCE_RUN_DIR)]
    if HELD_OUT_ROCKS:
        command += ["--held-out-rocks", *HELD_OUT_ROCKS]
    print("Komut:", " ".join(command))
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    loro_dir = None
    fold_dirs: list[Path] = []
    for line in process.stdout:
        print(line, end="")
        if line.startswith("run_dir="):
            fold_dir = Path(line.split("=", 1)[1].strip())
            fold_dirs.append(fold_dir)
            copy_run_to_drive(fold_dir)  # kat biter bitmez Drive'a
        elif line.startswith("loro_dir="):
            loro_dir = Path(line.split("=", 1)[1].strip())
    if process.wait() != 0:
        raise RuntimeError("LORO hata ile bitti; yukarıdaki çıktıya bak. Biten katlar Drive'da.")
    if loro_dir is None or not loro_dir.is_dir():
        raise RuntimeError("loro_dir= satırı bulunamadı")
    copy_run_to_drive(loro_dir)
    return loro_dir, fold_dirs


if REFERENCE_RUN_DIR is not None:
    REFERENCE_RUN_DIR = Path(REFERENCE_RUN_DIR)
    if not (REFERENCE_RUN_DIR / "metrics.json").is_file():
        raise FileNotFoundError(f"Referans koşuda metrics.json yok: {REFERENCE_RUN_DIR}")
if not (V2_LORO_DIR / "loro_summary.csv").is_file():
    raise FileNotFoundError(f"v2 LORO özeti yok: {V2_LORO_DIR / 'loro_summary.csv'}")
for tag, done_dir in DONE_LORO_DIRS.items():
    if tag in WD_TO_RUN:
        raise ValueError(f"wd {tag} hem DONE_LORO_DIRS'te hem WD_TO_RUN'da; birini seç")
    if not (done_dir / "loro_summary.csv").is_file():
        raise FileNotFoundError(f"wd {tag} toplu özeti yok: {done_dir / 'loro_summary.csv'}")
print("Hazır.")


## 7. LORO: her yay sertliği için 8 kat

`WD_TO_RUN` sırasıyla. Her sertlik ≈ 2 s 40 dk (T4). Oturum yetmezse: biten sertliği `WD_TO_RUN`'dan çıkar ve toplu klasörünü `DONE_LORO_DIRS`'e yaz; yarım kalan sertliği baştan koş (tek parça 8-kat özeti çıkar). `HELD_OUT_ROCKS` ile sadece kalan kayaçları koşmak da mümkün, ama o zaman özet tablo o kayaçlarla sınırlı kalır.

In [ ]:
LORO_DIRS: dict[str, Path] = {}
FOLD_DIRS: dict[str, list[Path]] = {}

for tag in WD_TO_RUN:
    print(f"\n==================== weight_decay {tag} ====================")
    loro_dir, fold_dirs = run_loro(WD_CONFIGS[tag])
    LORO_DIRS[tag] = loro_dir
    FOLD_DIRS[tag] = fold_dirs
    print(f"wd {tag}: {len(fold_dirs)} kat + toplu klasör Drive'da: {DRIVE_RUN_ROOT / loro_dir.name}")


## 8. Sonuç: kayaç × yay sertliği

`unseen` = modelin hiç görmediği kayaçtaki hata; `seen` = aynı modelin gördüğü kayaçların test payındaki hatası. Normalize sütunlar her katın kendi cetveliyle hesaplanır (7 eğitim kayacından). Ölçütler sonuç görülmeden yazıldı; hücre bunları otomatik uygular.

In [ ]:
import pandas as pd
from IPython.display import Image, display

# Önceki oturumda biten sertlikler (Ayarlar: DONE_LORO_DIRS) önce, bu oturumda koşulanlar sonra.
LORO_DIRS = {**DONE_LORO_DIRS, **LORO_DIRS}

COLUMNS = ["unseen_rmse_mean_normalized", "seen_rmse_mean_normalized",
           "unseen_rmse_mean_original", "best_val_loss"]
summaries = {"0 (v2)": pd.read_csv(V2_LORO_DIR / "loro_summary.csv")}
for tag, loro_dir in LORO_DIRS.items():
    summaries[tag] = pd.read_csv(loro_dir / "loro_summary.csv")

frames = []
for tag, frame in summaries.items():
    part = frame.set_index("held_out_rock")[COLUMNS].copy()
    part.columns = pd.MultiIndex.from_product([[tag], COLUMNS])
    frames.append(part)
table = pd.concat(frames, axis=1)

print("Kayaç × yay sertliği — görülmemiş kayaç normalize RMSE (1.0 = ortalamayı söyleyen model)")
unseen = table.xs("unseen_rmse_mean_normalized", axis=1, level=1)
display(unseen.round(3))
print("Aynı katların görülen kayaç normalize RMSE (yay eğitimi ne kadar bozdu?)")
display(table.xs("seen_rmse_mean_normalized", axis=1, level=1).round(3))
print("best_val_loss (normalize MSE, kök alınmamış)")
display(table.xs("best_val_loss", axis=1, level=1).round(5))

# Önceden yazılmış ölçütler (spec: daily-sessions/2026-09-17_0112.md)
v2_unseen = unseen["0 (v2)"]
for tag in LORO_DIRS:
    common = unseen[tag].dropna().index.intersection(v2_unseen.dropna().index)
    improved = int((unseen.loc[common, tag] < v2_unseen.loc[common]).sum())
    below_one = int((unseen.loc[common, tag] < 1.0).sum())
    seen_max = float(table.loc[common, (tag, "seen_rmse_mean_normalized")].max())
    closed = improved >= 5 and below_one >= 4 and seen_max <= 0.10
    print(f"wd {tag}: v2'ye göre düşen kayaç {improved}/{len(common)}, "
          f"1.0 altı {below_one}/{len(common)}, seen max {seen_max:.3f} -> "
          f"{'B açığı KAPANDI' if closed else 'B açığı kapanmadı (sayılarla raporla)'}")

for tag, loro_dir in LORO_DIRS.items():
    print(f"\n===== weight_decay {tag}: {loro_dir.name} =====")
    for name in ("loro_rmse_normalized_by_rock.png", "loro_unseen_rmse_normalized_heatmap.png",
                 "loro_unseen_rmse_boxplot.png"):
        path = loro_dir / "plots" / name
        if path.is_file():
            print(name)
            display(Image(filename=str(path)))
    for fold_dir in FOLD_DIRS.get(tag, []):
        print(fold_dir.name)
        display(Image(filename=str(fold_dir / "plots" / "loss_curve.png")))
